**Vision-Based Gaze Estimation | Presented by: Reza Saadatyar (2026)**

**03 | Classical Gaze Estimation Baselines**

![Classical gaze-estimation workflow](../../images/03_classical_gaze_estimation_baselines.svg)

**Objective:** Compare a training-derived mean-gaze reference with Histogram of Oriented Gradients (HOG) and regularised Ridge regression.

**Data:** Notebook 02 provides 7,500 normalised left-eye images (36 × 60 pixels), 3D unit gaze vectors and 15 fixed participant folds. Each fold contains 6,000 Training, 1,000 inner Validation and 500 reserved Test images.

**Evaluation:** Select Ridge regularisation using inner Validation only. The default run does not score outer Test. The study-defined subset is not the official MPIIGaze benchmark.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess
repo_url="https://github.com/RezaSaadatyar/Eye-Tracking-Neurocognitive-Intelligence.git"
repo_path=Path("/content")/repo_url.rsplit("/",1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git","clone","-q",repo_url,str(repo_path)],check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}

**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root=setup_colab("vision_based_gaze_estimation")

# ---------------------------- 2. Standard Library ----------------------------
import sys

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------ 4. Classical Modelling and Evaluation --------------------
from gaze_models import hog_descriptors, normalise_directions
from gaze_models import fit_ridge_and_predict
from gaze_geometry import angular_error_degrees
from mpiigaze_data import restore_indexed_eye_cache
from gaze_evaluation import load_image_cache, outer_fold_indices
from gaze_evaluation import person_angular_metrics

# ------------------- 5. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

# --------------------- 6. Output Directory and Setup -------------------------
out_dir=root/"vision_based_gaze_estimation"/"output"
cache_path=root/"data"/"processed"/"mpiigaze"/"02_normalised_eye_cache.npz"
plot_style(font_family="DejaVu Sans", title_size=12.5,
 title_weight="normal",label_size=11.5,label_weight="normal",
 tick_size=11,legend_size=10,figure_dpi=100,figure_size=(9,4))
print("Output directory:",out_dir)
print("Image cache:",cache_path)
print("Python:",sys.version.split()[0])

**02. Check Image–Label Identity and Fixed Fold Roles**

The sample index and local image cache originate from Notebook 02. They must agree on sample order and 3D vectors. A missing cache is reconstructed from the exact saved Stage-02 source rows, acquiring the official raw archive in Colab if necessary. No new subset or fold assignments are created.

In [ ]:
# ================ 02. Verify Source-Aligned Modelling Inputs =================

required=["02_sample_index.csv","02_fold_assignments.csv"]
missing=[x for x in required if not (out_dir/x).is_file()]
if missing:
    raise FileNotFoundError(f"Notebook 02 inputs missing: {missing}")
# Recover the exact Stage-02 sample rows if this runtime is new.
if not cache_path.is_file():
    restore_indexed_eye_cache(
        root / "data" / "raw" / "mpiigaze",
        out_dir / "02_sample_index.csv", cache_path)
index,images,targets=load_image_cache(
 out_dir/"02_sample_index.csv",cache_path)
folds=pd.read_csv(out_dir/"02_fold_assignments.csv",
 dtype={"participant_id":str,"outer_test_person":str,"fold_id":str})
if folds.empty or folds.duplicated(["fold_id","participant_id"]).any():
    raise ValueError("Missing or repeated outer-fold participant assignments")
print("Source-aligned images:",len(index))
print("Independent people:",index.participant_id.nunique())
print("Fixed outer folds:",folds.fold_id.nunique())

**03. Compute Classical Eye-Image Descriptors**

HOG describes local image-gradient orientations using six orientation bins, 6 × 6-pixel cells and one cell per block. For a 36 × 60 image, this produces 360 features (6 × 10 × 6). Descriptors use only pixels; neither gaze labels nor participant identifiers are features. The Ridge feature scaler is fitted separately on each fold's Training data.

In [ ]:
# ================== 03. Compute Source-Only HOG Descriptors ==================

features=hog_descriptors(images)
if len(features)!=len(index) or not np.isfinite(features).all():
    raise ValueError("HOG arrays do not match input sample identity")
print("HOG feature matrix:",features.shape)
print("No target or participant ID enters the image descriptor")

**04. Fit and Select Models Within Each Outer Fold**

The mean-direction reference normalises the mean Training gaze vector. HOG + Ridge predicts three gaze components from standardised descriptors, then normalises the predictions. Candidate regularisation strengths are 1, 10 and 100, selected by minimum inner Validation mean angular error. For unit target and predicted directions, angular error is the inverse cosine of their clipped dot product, converted to degrees. Outer Test does not guide fitting or selection.

In [ ]:
# ==================== 04. Fit Nested Classical Baselines =====================

# -------------------------- 1. Candidate Design ------------------------------
SEED = 17
RIDGE_ALPHAS = (1., 10., 100.)
RUN_OUTER_TEST = False  # Development on inner Validation by default.
if not RUN_OUTER_TEST:
    previously_scored = [
        name for name in ("03_baseline_outer_predictions.csv",
                          "03_baseline_fold_metrics.csv")
        if (out_dir / name).is_file()]
    if previously_scored:
        raise RuntimeError(
            "Saved outer-Test files exist in this checkout. Preserve those "
            "results and use a clean development checkout: "
            + str(previously_scored))
outer_rows, inner_rows = [], []

# ---------------------- 2. Train and Select Within Fold ----------------------
for fold_id in sorted(folds.fold_id.unique()):
    role = outer_fold_indices(index, folds, fold_id)
    i_train, i_val, i_test = (role[k] for k in
                              ("train", "validation", "test"))
    train_mean = normalise_directions(
        targets[i_train].mean(axis=0)[None])[0]
    prior_val = np.tile(train_mean, (len(i_val), 1))
    prior_error = float(angular_error_degrees(
        targets[i_val], prior_val).mean())
    inner_rows.append({
        "fold_id": fold_id, "model": "mean_direction",
        "alpha": np.nan, "selected": True,
        "inner_val_mean_angular_error_deg": prior_error,
        "n_inner_people": index.iloc[i_val].participant_id.nunique()})
    best_alpha, best_error = None, float("inf")
    for alpha in RIDGE_ALPHAS:
        candidate, pred_val = fit_ridge_and_predict(
            features[i_train], targets[i_train], features[i_val], alpha)
        error = float(angular_error_degrees(
            targets[i_val], pred_val).mean())
        inner_rows.append({
            "fold_id": fold_id, "model": "hog_ridge", "alpha": alpha,
            "selected": False, "inner_val_mean_angular_error_deg": error,
            "n_inner_people": index.iloc[i_val].participant_id.nunique()})
        if error < best_error:
            best_alpha, best_error = alpha, error
    inner_rows[-len(RIDGE_ALPHAS) + RIDGE_ALPHAS.index(best_alpha)][
        "selected"] = True

    if RUN_OUTER_TEST:
        selected, pred_ridge = fit_ridge_and_predict(
            features[i_train], targets[i_train],
            features[i_test], best_alpha)
        prior_test = np.tile(train_mean, (len(i_test), 1))
        for name, prediction in (("mean_direction", prior_test),
                                 ("hog_ridge", pred_ridge)):
            rows = index.iloc[i_test][["sample_id", "participant_id"]].copy()
            rows["fold_id"] = fold_id
            rows["model"] = name
            rows[["target_x", "target_y", "target_z"]] = targets[i_test]
            rows[["pred_x", "pred_y", "pred_z"]] = prediction
            rows["selected_inner_alpha"] = (
                best_alpha if name == "hog_ridge" else np.nan)
            outer_rows.append(rows)
    print(fold_id, "inner Validation Ridge alpha:", best_alpha)

inner_validation = pd.DataFrame(inner_rows)
print("Outer Test scoring enabled:", RUN_OUTER_TEST)
print("Inner candidate/fold rows:", len(inner_validation))

**05. Compare Participant-Independent Inner Validation**

Summarise mean angular errors of the two selected procedures across the 15 outer-fold configurations. These inner Validation scores support development; repeated participants across configurations make fold scores dependent. They are not final unseen-person Test estimates.

In [ ]:
# =================== 05. Summarise Inner Validation Error ====================

selected = inner_validation.loc[inner_validation.selected]
summary = (selected.groupby("model")
           .inner_val_mean_angular_error_deg
           .agg(n_folds="count", mean="mean", median="median", maximum="max")
           .reset_index())
display(summary)
print("All scored development observations are from inner Validation.")
if RUN_OUTER_TEST:
    outer_predictions = pd.concat(outer_rows, ignore_index=True)
    recorded, per_person = person_angular_metrics(outer_predictions)
    if per_person.groupby("model").participant_id.nunique().min() != (
            index.participant_id.nunique()):
        raise ValueError("Missing outer Test participant predictions")
    print("Outer Test per-person errors (secondary report):")
    display(per_person)

**06. Save Classical Development and Optional Outer-Test Evidence**

The standard run stores inner-Validation results and a comparison figure without accessing Test labels. An explicitly requested full outer-LOPO run also stores Test predictions. Once an outer Test has been examined, subsequent model changes are exploratory unless independently confirmed.

In [ ]:
# =================== 06. Save Classical Modelling Evidence ===================

inner_validation.to_csv(
    out_dir / "03_inner_validation_metrics.csv", index=False)
configuration = selected[["fold_id", "model", "alpha"]].copy()
configuration = configuration.rename(columns={"alpha": "selected_alpha"})
configuration["run_mode"] = ("full_outer_lopo" if RUN_OUTER_TEST
                             else "development_inner_validation")
configuration["outer_fold_count"] = folds.fold_id.nunique()
configuration["test_used_for_selection"] = False
configuration.to_csv(out_dir / "03_run_configuration.csv", index=False)
fig, ax = plt.subplots(figsize=(9.5, 4.5))
for name, piece in selected.groupby("model"):
    ax.plot(piece.fold_id, piece.inner_val_mean_angular_error_deg,
            marker="o", label=name)
ax.set(title="Classical gaze models: inner Validation across outer folds",
       xlabel="Outer fold (Test labels not scored)",
       ylabel="Inner Validation angular error (degrees)")
ax.tick_params(axis="x", rotation=55)
ax.legend()
ax.grid(alpha=.2)
fig.tight_layout()
save_figure(fig, out_dir / "03_baseline_comparison.png")
plt.show()
if RUN_OUTER_TEST:
    outer_predictions.to_csv(
        out_dir / "03_baseline_outer_predictions.csv", index=False)
    per_person.to_csv(out_dir / "03_baseline_fold_metrics.csv", index=False)

**07. Interpretation and Output Download**

The development stage provides inner-Validation evidence and no outer-Test score. Full participant-held-out LOPO evaluation can be enabled after the modelling approach has been fixed. The next notebook uses the same participant folds for CNN development.

In [ ]:
# ================ 07. Interpret Classical Results and Export =================

needed = ["03_inner_validation_metrics.csv",
          "03_run_configuration.csv", "03_baseline_comparison.png"]
if RUN_OUTER_TEST:
    needed += ["03_baseline_outer_predictions.csv",
               "03_baseline_fold_metrics.csv"]
missing = [x for x in needed if not (out_dir / x).is_file()]
if missing:
    raise FileNotFoundError(f"Stage-03 outputs missing: {missing}")
print("Inner Validation candidate/fold rows:", len(inner_validation))
print("Outer Test scored:", RUN_OUTER_TEST)
print("Next: 04_cnn_and_transfer_learning.ipynb")
(out_dir / "03_run_configuration.json").unlink(missing_ok=True)
download_outputs_from_colab("vision_based_gaze_estimation/output", "03")